# Lab 13: The fact store

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/prekshapraveen/Documents/lantern-lab2-submission.nosync | LLM mode: replay


In [2]:
import duckdb
Path("data/work").mkdir(parents=True, exist_ok=True)
con = duckdb.connect("data/work/facts.duckdb")
con.execute("DROP INDEX IF EXISTS f_lookup")   # rerun-safe: the index first, then the table
con.execute("DROP TABLE IF EXISTS facts")

From the slide, the fact table:

```sql
CREATE TABLE facts (
  fact_id     VARCHAR PRIMARY KEY, -- sha1 of the key
  deal_id     VARCHAR NOT NULL,    -- 'public' | deal
  entity_id   VARCHAR NOT NULL,    -- CIK or deal id
  concept     VARCHAR,             -- us-gaap:Revenues
  metric      VARCHAR NOT NULL,    -- vocab: revenue
  value       DOUBLE  NOT NULL,
  unit        VARCHAR NOT NULL,    -- USD, shares, pure
  period_type VARCHAR,             -- duration|instant
  period_start DATE, period_end DATE,
  decimals    INTEGER,
  tier        TINYINT NOT NULL,    -- 1..3 only here
  source_type VARCHAR NOT NULL,
  doc_id      VARCHAR NOT NULL,
  citation    JSON    NOT NULL,    -- kind + locator
  extractor   VARCHAR,
  lineage     JSON                 -- parents, sha256
);
CREATE INDEX f_lookup ON facts(deal_id, metric,
                               period_end);
```

In [3]:
# Run the slide's DDL, one statement at a time
DDL = """CREATE TABLE facts (
  fact_id     VARCHAR PRIMARY KEY, -- sha1 of the key
  deal_id     VARCHAR NOT NULL,    -- 'public' | deal
  entity_id   VARCHAR NOT NULL,    -- CIK or deal id
  concept     VARCHAR,             -- us-gaap:Revenues
  metric      VARCHAR NOT NULL,    -- vocab: revenue
  value       DOUBLE  NOT NULL,
  unit        VARCHAR NOT NULL,    -- USD, shares, pure
  period_type VARCHAR,             -- duration|instant
  period_start DATE, period_end DATE,
  decimals    INTEGER,
  tier        TINYINT NOT NULL,    -- 1..3 only here
  source_type VARCHAR NOT NULL,
  doc_id      VARCHAR NOT NULL,
  citation    JSON    NOT NULL,    -- kind + locator
  extractor   VARCHAR,
  lineage     JSON                 -- parents, sha256
);
CREATE INDEX f_lookup ON facts(deal_id, metric,
                               period_end);"""
for stmt in [s for s in DDL.split(";") if s.strip()]:
    con.execute(stmt)

In [4]:
# From the slide: Seller financials from xlsx: tier-3 facts with cell citations
from openpyxl import load_workbook

def xlsx_facts(path, sheet, row_map, deal_id, doc_id):
    wb = load_workbook(path, data_only=True,
                       read_only=True)   # values only
    rows = wb[sheet].iter_rows()
    header = [c.value for c in next(rows)]
    for row in rows:
        label = row[0].value
        if label not in row_map:         # vocab mapping
            continue
        for j, cell in enumerate(row[1:], start=1):
            ref = f"{sheet}!{cell.coordinate}"
            if cell.value is None:       # fail loudly
                raise ValueError(f"{ref}: empty or "
                                 "uncached formula")
            yield {"deal_id": deal_id, "tier": 3,
                   "metric": row_map[label],
                   "period": header[j],
                   "value": float(cell.value),
                   "citation": {"kind": "sheet_cell",
                                "doc_id": doc_id,
                                "cell": ref}}

In [5]:
row_map = {"Revenue": "revenue", "Adjusted EBITDA": "adjusted_ebitda"}
raw = list(xlsx_facts("data/dataroom/deal_a/raw/financials.xlsx", "P&L", row_map, "deal_a", sd.XLSX_DOC))
print(raw[2])
# The sheet is in USD millions: scale before facts meet the linter, which reads "$41.2 million" as 41.2e6
facts = sd.seller_facts()
assert all(any(f["citation"]["cell"] == r["citation"]["cell"] and abs(f["value"] - r["value"] * 1e6) < 1 for f in facts) for r in raw)
print(len(facts), "seller facts, scaled; e.g.", [(f["metric"], f["period"], f["value"]) for f in facts if f["metric"] == "revenue"])

{'deal_id': 'deal_a', 'tier': 3, 'metric': 'revenue', 'period': 'FY2025', 'value': 41.2, 'citation': {'kind': 'sheet_cell', 'doc_id': 'DR-deal_a-004', 'cell': 'P&L!D2'}}
17 seller facts, scaled; e.g. [('revenue', 'FY2023', 26400000.0), ('revenue', 'FY2024', 33070000.0), ('revenue', 'FY2025', 41200000.0)]


In [6]:
from datetime import date
def fiscal_bounds(label, fye_month=6):          # Wren's fiscal year ends June 30
    y = int(label[2:]); return date(y - 1, fye_month + 1, 1).isoformat(), date(y, fye_month, 30).isoformat()
rows = []
for f in facts:
    start, end = fiscal_bounds(f["period"])
    rows.append((f["fact_id"], "deal_a", "deal_a", None, f["metric"], f["value"], f["unit"], "duration", start, end, None,
                 f["tier"], "dataroom", f["citation"]["doc_id"], json.dumps(f["citation"]), "openpyxl", json.dumps({"parents": f["citation"].get("parents", [])})))
con.executemany("INSERT INTO facts VALUES (" + ",".join(["?"] * 17) + ")", rows)
con.execute("SELECT metric, period_end, value, tier FROM facts WHERE deal_id = 'deal_a' ORDER BY metric, period_end").df()

,metric,period_end,value,tier
0,adjusted_ebitda,2023-06-30,6.000000e+06,3
1,adjusted_ebitda,2024-06-30,7.900000e+06,3
2,adjusted_ebitda,2025-06-30,9.900000e+06,3
3,arr,2025-06-30,4.800000e+07,3
4,ebitda,2023-06-30,5.100000e+06,3
5,ebitda,2024-06-30,6.400000e+06,3
6,ebitda,2025-06-30,7.500000e+06,3
7,gross_logo_churn,2025-06-30,7.000000e-02,3
8,gross_profit,2023-06-30,1.900000e+07,3
9,gross_profit,2024-06-30,2.410000e+07,3


In [7]:
# From the slide: Citations for XBRL facts: a bbox if you can, an element if you cannot
def xbrl_citation(fact, label_map_hits, doc_id):
    """Prefer the PDF cell CS1 Part 11 matched."""
    hit = label_map_hits.get((fact["concept"],
                              fact["period_end"]))
    if hit:                        # rendered + matched
        return {"kind": "pdf_bbox", "doc_id": doc_id,
                "page": hit["page"], "bbox": hit["bbox"],
                "block_id": hit["block_id"]}
    if fact.get("element_id"):     # ix:nonFraction id
        return {"kind": "ixbrl_element", "doc_id": doc_id,
                "element_id": fact["element_id"]}
    return {"kind": "ixbrl_context", "doc_id": doc_id,
            "concept": fact["concept"],
            "context_ref": fact["context_ref"],
            "unit_ref": fact.get("unit_ref")}

In [8]:
fact = {"concept": "us-gaap:Revenues", "period_end": "2025-12-31", "context_ref": "c-FY2025", "unit_ref": "usd"}
print(xbrl_citation(fact, {("us-gaap:Revenues", "2025-12-31"): {"page": 41, "bbox": [72, 410, 540, 424], "block_id": "p0041_b007"}}, "0000000001-25-000101"))
print(xbrl_citation(fact, {}, "0000000001-25-000101"))   # no CS1 match and no element id -> ixbrl_context

{'kind': 'pdf_bbox', 'doc_id': '0000000001-25-000101', 'page': 41, 'bbox': [72, 410, 540, 424], 'block_id': 'p0041_b007'}
{'kind': 'ixbrl_context', 'doc_id': '0000000001-25-000101', 'concept': 'us-gaap:Revenues', 'context_ref': 'c-FY2025', 'unit_ref': 'usd'}


**Exercise 13.** Add the KPIs sheet (NRR, gross logo churn) with the right scale for each row. Which KPI would break if you scaled the whole sheet by 1e6?

In [9]:
KPI_MAP = {"ARR (USD millions)": "arr",
           "Net revenue retention": "nrr",
           "Gross logo churn": "gross_logo_churn"}
SCALE = {"arr": 1e6, "nrr": 1.0, "gross_logo_churn": 1.0}
kpis = [{**f, "value": f["value"] * SCALE[f["metric"]]}
        for f in xlsx_facts("data/dataroom/deal_a/raw/financials.xlsx",
                            "KPIs", KPI_MAP, "deal_a", sd.XLSX_DOC)]
for f in kpis:
    print(f["metric"], f["period"], f["value"], f["citation"]["cell"])

arr FY2025 48000000.0 KPIs!B2
nrr FY2025 1.09 KPIs!B3
gross_logo_churn FY2025 0.07 KPIs!B4


### Exercise 13: answer

**Result:** with a scale chosen per row, the KPIs sheet loads as
`arr FY2025 48000000.0 (KPIs!B2)`, `nrr FY2025 1.09 (KPIs!B3)` and
`gross_logo_churn FY2025 0.07 (KPIs!B4)`.

**Which KPI breaks if the whole sheet is scaled by 1e6?** NRR and gross logo churn.
Only ARR is stated in USD millions, so only ARR needs ×1e6 (48 → 48,000,000).
NRR (1.09 = 109%) and gross logo churn (0.07 = 7%) are ratios: they have no "millions"
to undo. Scaling them by 1e6 would turn NRR into 1,090,000 and churn into 70,000,
which are meaningless, and the Lab 18 linter would then "verify" wrong numbers.

**Takeaway:** the scale belongs to each row (what kind of number it is), not to the
sheet. That's why the scale is kept per metric (`SCALE`, and the kit's `KPI_SCALE`).